# BioVision — training the vehicle specialist on CarDD

Fine-tunes YOLO segmentation on CarDD for the six damage classes: `dent`,
`scratch`, `crack`, `glass_shatter`, `lamp_broken`, `tire_flat`.

Runs on a free Colab T4. Inference in production is CPU-only.

## Why we train this ourselves

A public CarDD checkpoint has an unknown train/test split. Any leakage between it
and our evaluation split would make the per-class mAP table unverifiable — and that
table is this project's headline claim. An unverifiable specialist is worse than no
specialist, because it turns the honest answer into a confident wrong one.

So: **we control the split, and we pin the seed.** See `docs/DECISIONS.md` ADR-003.

## What this notebook must never do

- redistribute CarDD, or commit any part of it to the repository
- touch the test split before the final evaluation
- report a single averaged number instead of per-class metrics

## 0. Reproducibility

Everything downstream depends on these constants. Changing any of them invalidates
the published metrics, so they are declared once, here, and recorded in the run.

In [ ]:
SEED = 20260311
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

MODEL = "yolo11s-seg.pt"   # small: the production box is CPU-only
IMGSZ = 640
EPOCHS = 100
BATCH = 16

# The class order IS the contract with backend/src/biovision/models/specialists/
# vehicle_yolo.py::CARDD_CLASSES. The model emits integer ids; reordering this list
# silently relabels every prediction, so the loader below asserts they match.
CLASSES = ["dent", "scratch", "crack", "glass_shatter", "lamp_broken", "tire_flat"]

import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)
print(f"seed={SEED}  split={TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}  classes={CLASSES}")

In [ ]:
!pip -q install ultralytics

import torch
from ultralytics import YOLO

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Obtain CarDD

CarDD is distributed on request by its authors. Obtain your own copy through their
process and mount it below.

**Do not commit it, and do not re-host it.** This notebook consumes a copy; it never
publishes one.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CARDD_ROOT = Path("/content/drive/MyDrive/datasets/CarDD")
assert CARDD_ROOT.is_dir(), f"CarDD not found at {CARDD_ROOT}"

WORK = Path("/content/biovision")
WORK.mkdir(exist_ok=True)
print(sorted(p.name for p in CARDD_ROOT.iterdir()))

## 2. Convert COCO annotations to YOLO segmentation format

CarDD ships COCO-style polygons. YOLO wants one `.txt` per image with normalised
polygon coordinates.

The class mapping is asserted rather than assumed: if CarDD's category names do not
match `CLASSES` after normalisation, the conversion stops. A silent mismatch here
would train a model whose class 1 means something other than what the API says it
means, and nothing downstream would look wrong.

In [ ]:
import json
from collections import defaultdict


def normalise(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def convert_coco(annotation_file: Path, image_dir: Path, out_dir: Path) -> int:
    coco = json.loads(annotation_file.read_text())

    categories = {c["id"]: normalise(c["name"]) for c in coco["categories"]}
    found = sorted(set(categories.values()))
    assert found == sorted(CLASSES), (
        f"CarDD categories {found} do not match CLASSES {sorted(CLASSES)}. "
        "Fix the mapping before training -- a mismatch relabels every prediction."
    )
    class_index = {name: CLASSES.index(name) for name in CLASSES}

    images = {img["id"]: img for img in coco["images"]}
    per_image = defaultdict(list)
    for ann in coco["annotations"]:
        if ann.get("iscrowd"):
            continue
        per_image[ann["image_id"]].append(ann)

    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)

    written = 0
    for image_id, annotations in per_image.items():
        info = images[image_id]
        width, height = info["width"], info["height"]
        lines = []

        for ann in annotations:
            for polygon in ann.get("segmentation", []):
                if len(polygon) < 6:  # fewer than three points is not a polygon
                    continue
                coords = []
                for i in range(0, len(polygon), 2):
                    coords.append(min(1.0, max(0.0, polygon[i] / width)))
                    coords.append(min(1.0, max(0.0, polygon[i + 1] / height)))
                index = class_index[categories[ann["category_id"]]]
                lines.append(str(index) + " " + " ".join(f"{c:.6f}" for c in coords))

        if not lines:
            continue

        source = image_dir / info["file_name"]
        if not source.is_file():
            continue

        (out_dir / "labels" / f"{source.stem}.txt").write_text("\n".join(lines))
        (out_dir / "images" / source.name).write_bytes(source.read_bytes())
        written += 1

    return written


print("Point these at your CarDD layout, then run.")

## 3. Split — deterministic, and pinned

This is the cell that makes the reported metrics mean something. The split is a
function of the seed and the sorted filenames, so anyone with their own CarDD copy
reproduces exactly these three sets.

In [ ]:
import hashlib
import shutil


def split_dataset(converted: Path, target: Path) -> dict[str, int]:
    stems = sorted(p.stem for p in (converted / "images").iterdir())
    rng = random.Random(SEED)
    rng.shuffle(stems)

    n = len(stems)
    train_end = int(n * TRAIN_RATIO)
    val_end = train_end + int(n * VAL_RATIO)
    splits = {
        "train": stems[:train_end],
        "val": stems[train_end:val_end],
        "test": stems[val_end:],
    }

    for name, members in splits.items():
        for kind in ("images", "labels"):
            (target / name / kind).mkdir(parents=True, exist_ok=True)
        for stem in members:
            for source in (converted / "images").glob(f"{stem}.*"):
                shutil.copy2(source, target / name / "images" / source.name)
            label = converted / "labels" / f"{stem}.txt"
            if label.is_file():
                shutil.copy2(label, target / name / "labels" / label.name)

    # A fingerprint of the split itself. Record it with the metrics: if it ever
    # differs, the numbers are not comparable to the published ones.
    digest = hashlib.sha256(
        "|".join(f"{k}:{','.join(v)}" for k, v in sorted(splits.items())).encode()
    ).hexdigest()[:16]
    print(f"split fingerprint: {digest}")

    return {name: len(members) for name, members in splits.items()}


print("Run after conversion.")

In [ ]:
DATASET = WORK / "cardd_yolo"

data_yaml = WORK / "cardd.yaml"
data_yaml.write_text(
    f"""path: {DATASET}
train: train/images
val: val/images
test: test/images

# Order matters: it is the contract with vehicle_yolo.py::CARDD_CLASSES.
names:
"""
    + "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASSES))
)
print(data_yaml.read_text())

## 4. Train

Validation runs on the `val` split. The `test` split is not touched until section 5.

In [ ]:
model = YOLO(MODEL)

results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    deterministic=True,
    patience=20,
    project=str(WORK / "runs"),
    name="cardd_seg",
    # Damage is small and often low-contrast; heavy colour jitter hurts more than
    # it helps. Keep the defaults conservative and let the report say what happened.
    hsv_v=0.3,
    degrees=5.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
)

## 5. Evaluate on the held-out test split

Per class, deliberately. The literature finds `dent`, `scratch` and `crack` to be
the hard classes; if these numbers show the same, that is a correct result reported
honestly, not a defect to average away.

In [ ]:
best = WORK / "runs" / "cardd_seg" / "weights" / "best.pt"
trained = YOLO(str(best))

metrics = trained.val(data=str(data_yaml), split="test", imgsz=IMGSZ)

print("| Class | mAP@50 | mAP@50-95 | Precision | Recall |")
print("|---|---|---|---|---|")
for i, name in enumerate(CLASSES):
    p, r, m50, m5095 = metrics.seg.class_result(i)
    print(f"| {name} | {m50:.3f} | {m5095:.3f} | {p:.3f} | {r:.3f} |")
print(
    f"| **all** | {metrics.seg.map50:.3f} | {metrics.seg.map:.3f} | "
    f"{metrics.seg.mp:.3f} | {metrics.seg.mr:.3f} |"
)

## 6. Export

Copy `best.pt` to `backend/weights/cardd_yolo_seg.pt`, then:

1. publish it as a GitHub release artifact (the checkpoint, never the dataset);
2. record its SHA-256 in `backend/scripts/fetch_weights.py`;
3. paste the table above into README section 7.3 **verbatim**, weak rows included;
4. record the split fingerprint alongside it;
5. regenerate the golden set: `uv run python -m scripts.update_golden --review`.

In [ ]:
import hashlib

digest = hashlib.sha256(best.read_bytes()).hexdigest()
print(f"cardd_yolo_seg.pt\nsha256: {digest}\nsize:   {best.stat().st_size / 1e6:.1f} MB")